In [2]:
import pandas as pd
import numpy as np

# Set a random seed so our randomized data is reproducible
np.random.seed(42)
n_samples = 500

# Simulate different fungal strains and environmental conditions
strains = np.random.choice(['Pleurotus_ostreatus', 'Rhizophagus_irregularis', 'Trichoderma_harzianum'], n_samples)
conditions = np.random.choice(['Optimal_Moisture', 'Drought_Stress'], n_samples)

# Simulate nutrient uptake values (mg/g) using a normal distribution
p_uptake = np.random.normal(loc=15.0, scale=3.5, size=n_samples)
k_uptake = np.random.normal(loc=40.0, scale=8.0, size=n_samples)

# Inject some realistic "lab errors" (simulating a failed sensor reading)
p_uptake[np.random.choice(n_samples, 15, replace=False)] = np.nan

# Build the Pandas DataFrame
raw_data = pd.DataFrame({
    'Strain': strains,
    'Environment': conditions,
    'Phosphorus_Uptake_mg': p_uptake,
    'Potassium_Uptake_mg': k_uptake
})

# Display the first 5 rows of our new dataset
raw_data.head()

,Strain,Environment,Phosphorus_Uptake_mg,Potassium_Uptake_mg
0,Trichoderma_harzianum,Drought_Stress,17.156373,35.408884
1,Pleurotus_ostreatus,Drought_Stress,13.149466,36.284871
2,Trichoderma_harzianum,Optimal_Moisture,10.518981,29.388362
3,Trichoderma_harzianum,Drought_Stress,10.444388,49.236674
4,Pleurotus_ostreatus,Drought_Stress,15.177949,28.372588


In [8]:
# Check for missing data points
print("Missing values before cleaning:")
print(raw_data.isnull().sum())

# Drop the corrupted rows
clean_data = raw_data.dropna()

# Ensure no impossible negative nutrient uptake values exist
clean_data = clean_data[(clean_data['Phosphorus_Uptake_mg'] > 0) & (clean_data['Potassium_Uptake_mg'] > 0)]

print(f"\nTotal valid samples after cleaning: {len(clean_data)} out of {n_samples}")

Missing values before cleaning:
Strain                   0
Environment              0
Phosphorus_Uptake_mg    15
Potassium_Uptake_mg      0
dtype: int64

Total valid samples after cleaning: 485 out of 500


In [9]:
# Group the data to analyze performance under different stress conditions
summary_stats = clean_data.groupby(['Environment', 'Strain'])[['Phosphorus_Uptake_mg', 'Potassium_Uptake_mg']].mean()

print("Mean Nutrient Uptake by Strain and Condition:\n")
print(summary_stats)

# Isolate the data specifically for Drought Stress
drought_data = summary_stats.loc['Drought_Stress']

# Find the best performing strain for Phosphorus uptake under stress
best_p_strain = drought_data['Phosphorus_Uptake_mg'].idxmax()

print(f"\nInsight: Under Drought Stress conditions, the strain with the most resilient Phosphorus uptake is {best_p_strain}.")

Mean Nutrient Uptake by Strain and Condition:

                                          Phosphorus_Uptake_mg  \
Environment      Strain                                          
Drought_Stress   Pleurotus_ostreatus                 15.299943   
                 Rhizophagus_irregularis             16.006632   
                 Trichoderma_harzianum               14.715162   
Optimal_Moisture Pleurotus_ostreatus                 14.833879   
                 Rhizophagus_irregularis             15.416823   
                 Trichoderma_harzianum               15.555186   

                                          Potassium_Uptake_mg  
Environment      Strain                                        
Drought_Stress   Pleurotus_ostreatus                39.689889  
                 Rhizophagus_irregularis            38.604793  
                 Trichoderma_harzianum              42.979121  
Optimal_Moisture Pleurotus_ostreatus                40.644670  
                 Rhizophagus_irregularis